<a href="https://colab.research.google.com/github/viraj97-sl/ai-ml-ds-learning-hub/blob/master/05_Hands_On_Labs/notebooks/03_data_visualization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 03: Data Visualization Mastery

> **Goal:** Learn to choose the right chart for the right question, and make charts that communicate clearly.

**What you'll learn:**
1. Matplotlib fundamentals (the engine under everything)
2. Seaborn for statistical plots
3. Plotly for interactive charts
4. Chart selection guide
5. Publication-quality formatting

**Time:** ~2.5 hours

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Style settings
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
COLORS = ['#2196F3', '#FF5722', '#4CAF50', '#9C27B0', '#FF9800']

# Load datasets
titanic = pd.read_csv('https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv')
tips = sns.load_dataset('tips')
iris = sns.load_dataset('iris')
flights = sns.load_dataset('flights')

print('Datasets loaded!')
print(f'Titanic: {titanic.shape}, Tips: {tips.shape}, Iris: {iris.shape}')

## Part 1: Chart Selection Guide

| Question | Chart Type |
|----------|------------|
| Distribution of one variable | Histogram, KDE, Boxplot, Violin |
| Compare distributions across groups | Boxplot, Violin, Ridge plot |
| Relationship between 2 numeric variables | Scatter, Hexbin, 2D KDE |
| Trend over time | Line chart |
| Part of a whole | Pie (max 5 categories), Stacked bar |
| Ranking comparison | Horizontal bar |
| Correlation matrix | Heatmap |
| Many variable relationships | Pairplot |
| Geography | Choropleth, Scatter map |

## Part 2: Matplotlib Fundamentals

In [ ]:
# Understanding fig, axes — the key to matplotlib
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
# fig = the whole figure (canvas)
# axes = array of subplots, shape (2, 3)

# ── Plot 1: Histogram ─────────────────────────────────────────
ax = axes[0, 0]
ax.hist(titanic['Age'].dropna(), bins=30, color=COLORS[0], alpha=0.8, edgecolor='white')
ax.axvline(titanic['Age'].median(), color='red', linestyle='--', label=f'Median: {titanic["Age"].median():.0f}')
ax.set_xlabel('Age')
ax.set_ylabel('Count')
ax.set_title('Age Distribution')
ax.legend()

# ── Plot 2: Bar chart ─────────────────────────────────────────
ax = axes[0, 1]
survival_by_class = titanic.groupby('Pclass')['Survived'].mean() * 100
bars = ax.bar(['1st Class', '2nd Class', '3rd Class'], survival_by_class,
               color=COLORS[:3], alpha=0.8, edgecolor='white')
# Add value labels on bars
for bar, val in zip(bars, survival_by_class):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val:.1f}%', ha='center', va='bottom', fontweight='bold')
ax.set_ylabel('Survival Rate (%)')
ax.set_title('Survival Rate by Class')
ax.set_ylim(0, 80)

# ── Plot 3: Scatter plot ──────────────────────────────────────
ax = axes[0, 2]
survived = titanic[titanic['Survived'] == 1]
not_survived = titanic[titanic['Survived'] == 0]
ax.scatter(not_survived['Age'], not_survived['Fare'], alpha=0.4, s=15,
           color=COLORS[1], label='Did not survive')
ax.scatter(survived['Age'], survived['Fare'], alpha=0.6, s=15,
           color=COLORS[2], label='Survived')
ax.set_xlabel('Age')
ax.set_ylabel('Fare')
ax.set_title('Age vs Fare by Survival')
ax.legend(markerscale=2)
ax.set_ylim(0, 300)  # Trim extreme outliers

# ── Plot 4: Box plot ──────────────────────────────────────────
ax = axes[1, 0]
data_by_class = [titanic[titanic['Pclass']==c]['Fare'].dropna() for c in [1,2,3]]
bp = ax.boxplot(data_by_class, labels=['1st', '2nd', '3rd'],
                patch_artist=True, notch=True)
for patch, color in zip(bp['boxes'], COLORS):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax.set_xlabel('Passenger Class')
ax.set_ylabel('Fare')
ax.set_title('Fare Distribution by Class')
ax.set_ylim(0, 300)

# ── Plot 5: Stacked bar (proportional) ───────────────────────
ax = axes[1, 1]
pivot = titanic.pivot_table(values='Survived', index='Pclass', columns='Sex', aggfunc='mean') * 100
pivot.plot(kind='bar', ax=ax, color=[COLORS[0], COLORS[1]], alpha=0.8, rot=0)
ax.set_xlabel('Passenger Class')
ax.set_ylabel('Survival Rate (%)')
ax.set_title('Survival Rate by Class and Sex')
ax.legend(title='Sex')

# ── Plot 6: Line chart (time series) ─────────────────────────
ax = axes[1, 2]
flights_pivot = flights.pivot(index='month', columns='year', values='passengers')
for year in [1949, 1952, 1955, 1958, 1960]:
    ax.plot(flights_pivot.index, flights_pivot[year], label=str(year), marker='o', ms=3)
ax.set_xlabel('Month')
ax.set_ylabel('Passengers')
ax.set_title('Monthly Passengers Over Years')
ax.legend(title='Year', fontsize=8)
ax.tick_params(axis='x', rotation=45)

fig.suptitle('Data Visualization Gallery', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('visualization_gallery.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved!')

## Part 3: Seaborn Statistical Plots

In [ ]:
# Seaborn's strength: statistical plots with minimal code
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Pairplot — shows ALL pairwise relationships at once
# (run separately due to size)
# sns.pairplot(iris, hue='species', diag_kind='kde')

# ── Violin plot ───────────────────────────────────────────────
sns.violinplot(data=titanic, x='Pclass', y='Age', hue='Survived',
               split=True, ax=axes[0, 0], palette='Set2')
axes[0, 0].set_title('Age Distribution by Class & Survival')
axes[0, 0].set_xlabel('Passenger Class')

# ── Heatmap — correlation matrix ─────────────────────────────
numeric_cols = ['Age', 'Fare', 'SibSp', 'Parch', 'Survived', 'Pclass']
corr = titanic[numeric_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))  # Hide upper triangle
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, ax=axes[0, 1], square=True, cbar_kws={'shrink': 0.8})
axes[0, 1].set_title('Correlation Matrix')

# ── KDE plot — smooth distribution ───────────────────────────
sns.kdeplot(data=titanic, x='Age', hue='Survived', fill=True,
            alpha=0.5, ax=axes[0, 2], palette='Set1')
axes[0, 2].set_title('Age Distribution by Survival (KDE)')

# ── Categorical plot — multiple categories ────────────────────
sns.barplot(data=tips, x='day', y='total_bill', hue='sex',
            ax=axes[1, 0], palette='muted', capsize=0.1)
axes[1, 0].set_title('Average Bill by Day and Gender')

# ── Regression plot ───────────────────────────────────────────
sns.regplot(data=tips, x='total_bill', y='tip', ax=axes[1, 1],
            scatter_kws={'alpha': 0.4, 's': 30}, line_kws={'color': 'red'})
axes[1, 1].set_title('Bill vs Tip with Regression Line')

# ── Heatmap — pivot table ─────────────────────────────────────
flights_pivot = flights.pivot(index='month', columns='year', values='passengers')
sns.heatmap(flights_pivot, cmap='YlOrRd', annot=False, fmt='d',
            ax=axes[1, 2], cbar_kws={'label': 'Passengers'})
axes[1, 2].set_title('Airline Passengers Heatmap')

plt.suptitle('Seaborn Statistical Plots', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## Part 4: Interactive Plotly Charts

In [ ]:
# Plotly makes charts interactive — hover, zoom, click

# Interactive scatter — hover to see passenger name
fig = px.scatter(
    titanic.dropna(subset=['Age']),
    x='Age', y='Fare',
    color='Survived',
    symbol='Sex',
    size='Pclass',
    hover_data=['Name', 'Pclass'],
    color_discrete_map={0: '#FF5722', 1: '#4CAF50'},
    title='Titanic: Age vs Fare (Interactive — hover over points!)',
    labels={'Survived': 'Survived?', 'Age': 'Passenger Age', 'Fare': 'Ticket Fare'},
    template='plotly_white',
)
fig.update_layout(legend_title_text='Status')
fig.show()

In [ ]:
# Animated chart — change over time
gapminder = px.data.gapminder()

fig = px.scatter(
    gapminder,
    x='gdpPercap', y='lifeExp',
    animation_frame='year',
    animation_group='country',
    size='pop',
    color='continent',
    hover_name='country',
    size_max=55,
    range_x=[100, 100000],
    range_y=[25, 90],
    log_x=True,
    title='GDP vs Life Expectancy Over Time (1952-2007)',
    template='plotly_white',
)
fig.show()

In [ ]:
# Subplot dashboard
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        'Survival Rate by Class',
        'Age Distribution by Survival',
        'Fare by Class (Box)',
        'Embarked Port Distribution',
    ]
)

# Plot 1: Bar
survival_pct = titanic.groupby('Pclass')['Survived'].mean() * 100
fig.add_trace(go.Bar(x=['1st', '2nd', '3rd'], y=survival_pct,
                     marker_color=['#2196F3', '#FF9800', '#FF5722'],
                     name='Survival %'), row=1, col=1)

# Plot 2: Histogram overlay
for survived, color, name in [(0, '#FF5722', 'Did Not Survive'), (1, '#4CAF50', 'Survived')]:
    ages = titanic[titanic['Survived'] == survived]['Age'].dropna()
    fig.add_trace(go.Histogram(x=ages, name=name, opacity=0.7,
                               marker_color=color, nbinsx=25), row=1, col=2)

# Plot 3: Box plots
for pclass, color in zip([1, 2, 3], ['#2196F3', '#FF9800', '#FF5722']):
    fares = titanic[titanic['Pclass'] == pclass]['Fare'].dropna()
    fig.add_trace(go.Box(y=fares, name=f'Class {pclass}',
                         marker_color=color), row=2, col=1)

# Plot 4: Pie chart
embarked_counts = titanic['Embarked'].value_counts()
fig.add_trace(go.Pie(labels=embarked_counts.index,
                     values=embarked_counts.values,
                     name='Embarked'), row=2, col=2)

fig.update_layout(
    title_text='Titanic Dashboard',
    title_font_size=20,
    height=700,
    template='plotly_white',
    showlegend=False,
)
fig.show()

## Part 5: Publication-Quality Formatting Tips

In [ ]:
# Before vs After: Making charts presentation-ready
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

data = titanic.groupby('Pclass')['Survived'].mean() * 100

# ── BEFORE: Default styling ───────────────────────────────────
ax1.bar([1, 2, 3], data)
ax1.set_title('BEFORE: Default Chart')

# ── AFTER: Publication quality ───────────────────────────────
colors = ['#1a9641', '#fdae61', '#d7191c']  # Meaningful color scale
bars = ax2.bar(['1st Class\n(Wealthy)', '2nd Class\n(Middle)', '3rd Class\n(Poor)'],
               data, color=colors, width=0.6, edgecolor='white', linewidth=1.5)

# Add value labels
for bar, val in zip(bars, data):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f'{val:.0f}%', ha='center', va='bottom',
             fontweight='bold', fontsize=12)

# Annotation for context
ax2.axhline(y=data.mean(), color='gray', linestyle='--', alpha=0.5)
ax2.text(2.5, data.mean() + 1, f'Average: {data.mean():.0f}%',
         ha='right', color='gray', fontsize=9)

# Remove chart junk
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

# Meaningful labels
ax2.set_ylabel('Percentage of passengers who survived', fontsize=11)
ax2.set_title('AFTER: Class Privilege Was Life or Death\nTitanic survival rates by passenger class',
              fontsize=12, loc='left', fontweight='bold')
ax2.set_ylim(0, 85)

# Source annotation
ax2.text(0, -0.12, 'Source: Titanic passenger records | n=891 passengers',
         transform=ax2.transAxes, fontsize=8, color='gray')

plt.tight_layout()
plt.show()

print('Key improvements:')
print('1. Meaningful colors (green=high, red=low)')
print('2. Value labels on bars')
print('3. Average reference line')
print('4. Removed top/right spines (chart junk)')
print('5. Descriptive title that tells the story')
print('6. Source attribution')

## Challenge: Build Your Own Dashboard

Create a 2x2 matplotlib figure (or Plotly dashboard) that answers these 4 questions about the Tips dataset:
1. Do people tip more on weekends vs weekdays?
2. Is there a relationship between table size and total bill?
3. Which meal (lunch/dinner) generates higher tips?
4. Does smoking status affect tip percentage?

Each subplot should have a clear title stating the answer, not just the question.